In [ ]:
pip install --upgrade pip

In [ ]:
pip install pandas scipy matplotlib tqdm torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [2]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
import torch
import torch.nn as nn
import numpy as np
import pickle
import pandas as pd
import torchvision.transforms as transforms
import torchvision.datasets as datasets
import math
import scipy.special
import random as rd
import torch.nn.functional as F
import torchvision.models as models
import matplotlib.pyplot as plt
from torchvision.models import VGG16_Weights
from tqdm import tqdm
import pickle
import torch.optim.lr_scheduler as lr_scheduler
from sgr_utils import *

print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0))

GPU Available: True
GPU Name: NVIDIA GeForce RTX 4060


### Loading cifar-10 dataset

In [3]:
# Define transforms for CIFAR-10 dataset
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
]) # imageNet stats by rgb channel

# Load CIFAR-10 dataset
train_dataset = datasets.CIFAR10(root="C:/Users/ejeme/Documents/python_repos/CIFAR/cifar_data", train=True, transform=transform, download=True)
test_dataset = datasets.CIFAR10(root="C:/Users/ejeme/Documents/python_repos/CIFAR/cifar_data", train=False, transform=transform, download=True)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=5, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=5, shuffle=False)

### Training vgg16 for cifar-10 classification task

In [8]:
# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Load Pretrained VGG16 Model
vgg16 = models.vgg16(weights=VGG16_Weights.DEFAULT)
# Modify classifier for CIFAR-10 (10 classes)
vgg16.classifier[6] = nn.Linear(4096, 10)
# Move model to GPU if available
vgg16 = vgg16.to(device)

In [ ]:
# Define Loss, Optimizer and lr scheduler
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(vgg16.parameters(), lr=1e-4)
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=2)

# Training Loop
num_epochs = 20
for epoch in range(num_epochs):
    vgg16.train()
    running_loss = 0
    correct = 0
    total = 0

    print('TRAINING EPOCH', epoch)
    for images, labels in tqdm(train_loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = vgg16(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
            
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")
    print(f"Train Accuracy: {100 * correct / total:.2f}%")

    # Evaluate Model at the end of current epoch
    vgg16.eval()
    correct = 0
    total = 0
    print('TESTING')
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = vgg16(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    print(f"Validation Accuracy: {100 * correct / total:.2f}%")

    scheduler.step(correct)
    torch.save(vgg16.state_dict(), "C:/Users/ejeme/Documents/python_repos/CIFAR/vgg16_cifar10_epoch"+str(epoch)+".pth")

TRAINING EPOCH 0


100%|██████████| 10000/10000 [17:03<00:00,  9.77it/s]


Epoch [1/20], Loss: 0.7362
Train Accuracy: 74.99%
TESTING
Test Accuracy: 81.75%
TRAINING EPOCH 1


100%|██████████| 10000/10000 [17:03<00:00,  9.78it/s]


Epoch [2/20], Loss: 0.4368
Train Accuracy: 85.74%
TESTING
Test Accuracy: 85.17%
TRAINING EPOCH 2


...

Epoch [13/20], Loss: 0.0024
Train Accuracy: 99.94%
TESTING
Test Accuracy: 91.44%
TRAINING EPOCH 13


100%|██████████| 10000/10000 [17:03<00:00,  9.77it/s]


Epoch [14/20], Loss: 0.0030
Train Accuracy: 99.93%
TESTING
Test Accuracy: 91.59%
TRAINING EPOCH 14


  4%|▎         | 370/10000 [00:37<16:27,  9.75it/s]


KeyboardInterrupt: 

Epoch 1 with 85% train and test accuracy is fine

### Retrieving Softmax Response, Predicted class and True class for all samples in train and in test sets

In [4]:
# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Load Pretrained VGG16 Model
vgg16 = models.vgg16(weights=VGG16_Weights.DEFAULT)
# Modify classifier for CIFAR-10 (10 classes)
vgg16.classifier[6] = nn.Linear(4096, 10)
# Move model to GPU if available
vgg16 = vgg16.to(device)

checkpoint = torch.load("C:/Users/ejeme/Documents/python_repos/CIFAR/vgg16_cifar10_epoch5.pth")
vgg16.load_state_dict(checkpoint)

<All keys matched successfully>

In [5]:
sgr_dico = prepare_sgr_dico(test_loader, model = vgg16, device = device, T = 10)
sgr_set = pd.DataFrame(sgr_dico)
pickle.dump(sgr_set, open('sgr_set','wb'))

100%|██████████| 2000/2000 [00:47<00:00, 42.29it/s]


In [7]:
sgr_set.sort_values('SR')

,y_true,y_pred,SR
5023,2.0,1.0,0.109236
5098,5.0,4.0,0.109463
5074,0.0,3.0,0.109504
8491,3.0,3.0,0.110281
6876,7.0,7.0,0.110312
...,...,...,...
2512,2.0,2.0,0.999997
2883,2.0,2.0,0.999998
5785,2.0,2.0,1.000000
2189,2.0,2.0,1.000000
